# 04 Experiments Over Baseline 1
In this notebook, I will be experimenting with models and fine-tuning that surpasses that of my baseline. The models I'll be experimenting with first will be those tailored to finance text, and they'll hopefully allow me to capture underlying tone and sentiment better than general-use models.

*Note 1: This notebook contains the simpler - almost baseline - models, and a lot of it simply experimentation with model-building and understanding the FinBERT architecture. The more prominent findings can be found in the next few notebooks.

*Note 2: This and all following notebooks will primarily be run in Google Colab, as I have signed up for Colab Pro and have access to faster GPUs than my local computer. The main reason why I couldn't use Colab previously was that WRDS could only be accessed from my local computer. However, with the data secured, that won't be a problem anymore.

# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date
from tqdm import tqdm
import os
import warnings

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

In [ ]:
!pip install gensim nltk evaluate transformers datasets

In [ ]:
from gensim.models import Word2Vec
import nltk
from nltk.util import ngrams
from nltk.tokenize import word_tokenize
from collections import defaultdict, Counter
import evaluate

import tensorflow as tf
from keras.models import Model
from keras.layers import Input, Embedding, Lambda, Dense
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

import torch
import torch.nn as nn

from datasets import Dataset, load_dataset

from transformers import BertTokenizer, BertModel, BertPreTrainedModel, BertForSequenceClassification
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, BertConfig
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Loading in full dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Changing directory to get the data
data_path = '/content/drive/MyDrive/MIDS DATASCI 266/MIDS DATASCI 266 Final Project'
os.chdir(data_path)
os.getcwd()

'/content/drive/MyDrive/MIDS DATASCI 266/MIDS DATASCI 266 Final Project'

In [ ]:
# Retrieving data
full_data_path = 'SNP500_Transcripts_Price_2015_to_2024.csv'
full_data = pd.read_csv(full_data_path, sep='|', index_col=0)

In [ ]:
# Limiting the data to only the text and the chosen target variable
model_df = full_data[['Text', 'Close_5_dir']].copy()
model_df['Close_5_dir'] = model_df['Close_5_dir'].astype(np.int64)

model_df

,Text,Close_5_dir
0,"Good afternoon. My name is Karen, and I'll be ...",0
1,"Ladies and gentlemen, thank you for standing b...",1
2,"Good day, ladies and gentlemen, and welcome to...",0
3,"Good morning, ladies and gentlemen, and welcom...",1
4,"Good morning, ladies and gentlemen, and welcom...",1
...,...,...
17228,"Good day, and thank you for standing by. Welco...",0
17229,Welcome to Lennar's Fourth Quarter Earnings Co...,0
17230,"Good afternoon, everyone. Welcome to NIKE, Inc...",0
17231,"Good day, and welcome to the FedEx Fiscal Year...",0


# Testing ModernBERT Baseline on Colab
For an initial test of the A100 GPU, I will run the exact same ModernBERT model as I did before and see how fast it can run it.

## Initial testing of ModernBERT
This first part will just be some initial testing of ModernBERT to confirm it works the same way on Colab as it does on my local computer.

In [ ]:
# Loading the ModernBERT model
mbert_checkpoint = "answerdotai/ModernBERT-base"
mbert_tokenizer = AutoTokenizer.from_pretrained(mbert_checkpoint)
mbert_model = AutoModel.from_pretrained(mbert_checkpoint)

# Getting the ModernBERT model with classification
mbert_classification_model = AutoModelForSequenceClassification.from_pretrained(mbert_checkpoint, num_labels=2)
mbert_classification_model.config.problem_type = "single_label_classification"

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Testing ModernBERT with the some earnings transcript data to make sure it can function properly
tscrpt_1_text = model_df['Text'][2]

# Getting the tokens
mbert_inputs_tscrpt_1 = mbert_tokenizer(tscrpt_1_text, return_tensors="pt")
print(mbert_inputs_tscrpt_1)
print(mbert_inputs_tscrpt_1.input_ids.shape)

# Getting what the tokenizer feeds in
mtokens_tscrpt_1 = mbert_tokenizer.tokenize(tscrpt_1_text)

Token indices sequence length is longer than the specified maximum sequence length for this model (9841 > 8192). Running this sequence through the model will result in indexing errors


{'input_ids': tensor([[50281,  8620,  1388,  ..., 14943,    32, 50282]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]])}
torch.Size([1, 9841])


In [ ]:
# Getting the ModernBERT outputs
mbert_outputs_tscrpt_1 = mbert_model(**mbert_inputs_tscrpt_1)
mbert_outputs_tscrpt_1

BaseModelOutput(last_hidden_state=tensor([[[-0.1602,  0.0031, -1.1323,  ...,  0.0533,  0.3774,  0.0036],
         [-0.4544,  0.7713,  0.3653,  ...,  1.0891, -0.9595, -0.2743],
         [-0.6195, -0.3943,  0.3787,  ...,  1.8312,  0.4605, -0.4912],
         ...,
         [-0.7410, -0.7556,  0.0879,  ...,  0.2443, -0.1505,  0.5807],
         [-1.1962,  0.1346,  0.5087,  ..., -0.4418, -0.3822, -1.7417],
         [ 0.1623, -0.0426, -0.0304,  ...,  0.0601,  0.2260,  0.1928]]],
       grad_fn=<NativeLayerNormBackward0>), hidden_states=None, attentions=None)

In [ ]:
# Getting the shape
mbert_outputs_tscrpt_1[0].shape

torch.Size([1, 9841, 768])

In [ ]:
# Getting the max length for ModernBERT
MAX_LEN = 8192

# Defining a function to tokenize the text
def tokenize_fn(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

## ModernBERT classifier model
Now I'll actually run our ModernBERT classifier model and see how fast it is.

In [ ]:
# Splitting the data using train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    model_df['Text'].tolist(),
    model_df['Close_5_dir'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=model_df['Close_5_dir']
)

# Converting to dictionaries
train_dict = {'text': X_train, 'label': y_train}
val_dict = {'text': X_val, 'label': y_val}

# Creating HuggingFace Datasets
train_dataset = Dataset.from_dict(train_dict)
val_dataset = Dataset.from_dict(val_dict)

In [ ]:
# Defining a function for computing the metrics
metric = evaluate.load('accuracy')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# Defining a preprocessing transcripts function for my earnings data
def preprocess_transcripts(data, tokenizer):
    texts = data['text']

    encoded = tokenizer.batch_encode_plus(
        texts,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    return {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'label': data['label']
    }

In [ ]:
# Creating a ModernBERT classification model
def fine_tune_classif_model_modernbert(classification_model,
                                       tokenizer,
                                       train_data,
                                       dev_data,
                                       layers_to_train=["classifier."],
                                       max_sequence_length=8192,
                                       batch_size=4,
                                       num_epochs=2,
                                       saved_output_dir="./modernbert_output_test"):

    # Getting the preprocessed data and setting the formats
    preprocessed_train_data = train_data.map(preprocess_transcripts, batched=True, fn_kwargs={'tokenizer': tokenizer})
    preprocessed_dev_data = dev_data.map(preprocess_transcripts, batched=True, fn_kwargs={'tokenizer': tokenizer})

    # Freezing all model parameters except those explicitly listed
    for name, param in classification_model.named_parameters():
        if not any(x in name for x in layers_to_train):
            param.requires_grad = False

    training_args = TrainingArguments(
        output_dir=saved_output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        report_to="none"
    )

    trainer = Trainer(
        model=classification_model,
        args=training_args,
        train_dataset=preprocessed_train_data,
        eval_dataset=preprocessed_dev_data,
        compute_metrics=compute_metrics
    )

    trainer.train()

In [ ]:
# Setting torch to high for better performance
torch.set_float32_matmul_precision('high')

In [ ]:
# Getting some smaller testing data to make sure things work properly
train_data_small = train_dataset.select(range(800))
val_data_small = val_dataset.select(range(200))

In [ ]:
# Calling the fine-tuning function and training on the model
fine_tune_classif_model_modernbert(
    classification_model=mbert_classification_model,
    tokenizer=mbert_tokenizer,
    train_data=train_data_small,
    dev_data=val_data_small,
    layers_to_train=["classifier."],
    max_sequence_length=MAX_LEN,
    batch_size=2,
    num_epochs=2,
    saved_output_dir="./modernbert_output_test_1"
)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.687383,0.545000
2,0.702900,0.688109,0.525000


It's very evident that this can run much faster than my local computer. Whereas it took me over an hour to run this ModernBERT testing, it took the A100 GPU roughly 15 minutes. The speed is definitely better from before, and it gives me confidence that I can train models successfully with Colab Pro.

# FinBERT Experiments
I'll now try the established FinBERT model from ProsusAI and experiment how it performs with the classification task using just the transcript data.

Note: This FinBERT model was pretrained for financial sentiment classification, so there's a softmax output of positive/negative/neutral. Since we only want a binary classification, I'll replace the classification head with my own two-class output.

## Loading in FinBERT and checking parameters
Firstly, we'll load in FinBERT and do some initial exploration with the model.

In [ ]:
# Loading the FinBERT model and tokenizer from Hugging face
finbert_model_name = "ProsusAI/finbert"
finbert_tokenizer = AutoTokenizer.from_pretrained(finbert_model_name)

# Overriding the original 3-class output to get our binary classes
finbert_classification_model = BertForSequenceClassification.from_pretrained(
    finbert_model_name,
    num_labels=2,
    ignore_mismatched_sizes=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ProsusAI/finbert and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Checking out the parameters
for name, param in finbert_classification_model.named_parameters():
    print(name, param.shape)

bert.embeddings.word_embeddings.weight torch.Size([30522, 768])
bert.embeddings.position_embeddings.weight torch.Size([512, 768])
bert.embeddings.token_type_embeddings.weight torch.Size([2, 768])
bert.embeddings.LayerNorm.weight torch.Size([768])
bert.embeddings.LayerNorm.bias torch.Size([768])
bert.encoder.layer.0.attention.self.query.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.self.query.bias torch.Size([768])
bert.encoder.layer.0.attention.self.key.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.self.key.bias torch.Size([768])
bert.encoder.layer.0.attention.self.value.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.self.value.bias torch.Size([768])
bert.encoder.layer.0.attention.output.dense.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.output.dense.bias torch.Size([768])
bert.encoder.layer.0.attention.output.LayerNorm.weight torch.Size([768])
bert.encoder.layer.0.attention.output.LayerNorm.bias torch.Size([768])
bert.encoder

Immediately, we run into the problem of FinBERT only being able to handle 512 tokens. This is a far cry from our full transcript lengths, and we'll definitely need to try different ways later to split the text up if we want to use this model.

## FinBERT with truncated text 1
To start, I'll simply try truncating my transcript text and see if the model can run that way.

In [ ]:
# Getting the max length for FinBERT
MAX_LEN = 512

# Splitting the data using train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    model_df['Text'].tolist(),
    model_df['Close_5_dir'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=model_df['Close_5_dir']
)

# Converting to dictionaries
train_dict = {'text': X_train, 'label': y_train}
val_dict = {'text': X_val, 'label': y_val}

# Creating HuggingFace Datasets
train_dataset = Dataset.from_dict(train_dict)
val_dataset = Dataset.from_dict(val_dict)

In [ ]:
# Defining a function for computing the metrics
metric = evaluate.load('accuracy')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# Defining a preprocessing transcripts function for my earnings data
def preprocess_transcripts(data, tokenizer):
    texts = data['text']

    encoded = tokenizer.batch_encode_plus(
        texts,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    return {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'label': data['label']
    }

In [ ]:
# Creating a general purpose classification model
def fine_tune_classif_model(classification_model,
                            tokenizer,
                            train_data,
                            dev_data,
                            layers_to_train=["classifier."],
                            max_sequence_length=8192,
                            batch_size=4,
                            num_epochs=2,
                            saved_output_dir="/tmp/temp_model"):

    # Getting the preprocessed data and setting the formats
    preprocessed_train_data = train_data.map(preprocess_transcripts, batched=True, fn_kwargs={'tokenizer': tokenizer})
    preprocessed_dev_data = dev_data.map(preprocess_transcripts, batched=True, fn_kwargs={'tokenizer': tokenizer})

    # Freezing all model parameters except those explicitly listed
    for name, param in classification_model.named_parameters():
        if not any(x in name for x in layers_to_train):
            param.requires_grad = False
        else:
            param.requires_grad = True

    training_args = TrainingArguments(
        output_dir=saved_output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        report_to="none",
        # load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True
    )

    trainer = Trainer(
        model=classification_model,
        args=training_args,
        train_dataset=preprocessed_train_data,
        eval_dataset=preprocessed_dev_data,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]
    )

    trainer.train()
    return trainer

In [ ]:
# Setting torch to high for better performance
torch.set_float32_matmul_precision('high')

### FinBERT w/ truncated text and dataset test
Firstly just testing if FinBERT works.

In [ ]:
# Getting some smaller testing data to make sure things work properly
train_data_small = train_dataset.select(range(800))
val_data_small = val_dataset.select(range(200))

In [ ]:
# Calling the fine-tuning function and training on the model
fine_tune_classif_model(
    classification_model=finbert_classification_model,
    tokenizer=finbert_tokenizer,
    train_data=train_data_small,
    dev_data=val_data_small,
    layers_to_train=["classifier."],
    max_sequence_length=MAX_LEN,
    batch_size=4,
    num_epochs=2,
    saved_output_dir="./finbert_truncation_test"
)

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.702936,0.505000
2,No log,0.703160,0.515000


Now that we confirmed it works on a small scale, let's scale things up with the full dataset and a larger batch size with more epochs.

### FinBERT w/ truncated text test 1.1
We'll try FinBERT with batch size of 8, num epochs of 10, and only unfreezing the classifier.

In [ ]:
# Calling the fine-tuning function and training on the model
fine_tune_classif_model(
    classification_model=finbert_classification_model,
    tokenizer=finbert_tokenizer,
    train_data=train_dataset,
    dev_data=val_dataset,
    layers_to_train=["classifier."],
    max_sequence_length=MAX_LEN,
    batch_size=8,
    num_epochs=10,
    saved_output_dir="./finbert_truncation_test"
)

Map:   0%|          | 0/13786 [00:00<?, ? examples/s]

Map:   0%|          | 0/3447 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.695300,0.693316,0.519872
2,0.693600,0.693680,0.520162
3,0.696600,0.692856,0.526545
4,0.692100,0.692293,0.536989
5,0.695700,0.693385,0.532927
6,0.692700,0.691851,0.535828
7,0.691400,0.691833,0.536989
8,0.693400,0.691471,0.536699
9,0.694600,0.691412,0.536408
10,0.692300,0.691400,0.536699


It's a good baseline for accuracy as it matches the majority class baseline (~53.7%), but let's if we can improve upon that by updating some layers to train.

Also, I'll return the trainer so we can get precision, recall, and f1 scores as well so we can make sure it's not just predicting the majority class.

### FinBERT w/ truncated text test 1.2
We'll try FinBERT with batch size of 8, num epochs of 10, and unfreezing the classifier, final pooler, and final encoder layers.

In [ ]:
# Calling the fine-tuning function and training on the model
f_trainer = fine_tune_classif_model(
    classification_model=finbert_classification_model,
    tokenizer=finbert_tokenizer,
    train_data=train_dataset,
    dev_data=val_dataset,
    layers_to_train=["classifier.", "bert.pooler.", "bert.encoder.layer.11."],
    max_sequence_length=MAX_LEN,
    batch_size=8,
    num_epochs=10,
    saved_output_dir="./finbert_truncation_test_2"
)

Map:   0%|          | 0/13786 [00:00<?, ? examples/s]

Map:   0%|          | 0/3447 [00:00<?, ? examples/s]

Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.690500,0.694165,0.543661
2,0.687900,0.694557,0.530606
3,0.687100,0.694481,0.535828
4,0.679000,0.699461,0.540760


In [ ]:
# Getting the predictions output
preprocessed_dev_data = val_dataset.map(preprocess_transcripts, batched=True, fn_kwargs={'tokenizer': finbert_tokenizer})
predictions_output = f_trainer.predict(preprocessed_dev_data)

# Getting the predicted class labels
preds = np.argmax(predictions_output.predictions, axis=1)
labels = predictions_output.label_ids

Map:   0%|          | 0/3447 [00:00<?, ? examples/s]

In [ ]:
# Getting the accuracy, precision, recall, and f1-scores
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

print("For the majority (up) class:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

For the majority (up) class:
Accuracy:  0.5408
Precision: 0.5495
Recall:    0.8005
F1 Score:  0.6517


In [ ]:
# Calling the fine-tuning function and training on the model
f_trainer = fine_tune_classif_model(
    classification_model=finbert_classification_model,
    tokenizer=finbert_tokenizer,
    train_data=train_dataset,
    dev_data=val_dataset,
    layers_to_train=["classifier.", "bert.pooler.", "bert.encoder.layer.11."],
    max_sequence_length=MAX_LEN,
    batch_size=8,
    num_epochs=10,
    saved_output_dir="./finbert_truncation_test_2"
)

Map:   0%|          | 0/13786 [00:00<?, ? examples/s]

Map:   0%|          | 0/3447 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.696700,0.690610,0.536699
2,0.697000,0.690318,0.536699
3,0.695000,0.690310,0.536699
4,0.690300,0.688999,0.543371
5,0.685800,0.714467,0.540760
6,0.685600,0.700135,0.539310
7,0.676300,0.707784,0.541921
8,0.670500,0.710782,0.538729
9,0.663000,0.716048,0.533507
10,0.655600,0.718585,0.534668


In [ ]:
# Getting the predictions output
preprocessed_dev_data = val_dataset.map(preprocess_transcripts, batched=True, fn_kwargs={'tokenizer': finbert_tokenizer})
predictions_output = f_trainer.predict(preprocessed_dev_data)

# Getting the predicted class labels
preds = np.argmax(predictions_output.predictions, axis=1)
labels = predictions_output.label_ids

Map:   0%|          | 0/3447 [00:00<?, ? examples/s]

In [ ]:
# Getting the accuracy, precision, recall, and f1-scores
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

print("For the majority (up) class:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

For the majority (up) class:
Accuracy:  0.5347
Precision: 0.5568
Recall:    0.6514
F1 Score:  0.6004


Overall, the accuracy wasn't great, and it did not beat baseline. Also, it appears that our model is definitely predicting Class 1 ("Up") more than Class 0 ("Down") due to the high recall. This could indicate there was imbalance in the dataset, but since we know there's not, this is quite strange. Let's try one more classification, this time unfreezing even more layers and seeing if there's improvement.

### FinBERT w/ truncated text test 1.3
We'll try FinBERT with batch size of 8, num epochs of 10, and unfreezing the classifier, final pooler, and encoder layers 8, 9, 10, and 11.

In [ ]:
# Calling the fine-tuning function and training on the model
f_trainer = fine_tune_classif_model(
    classification_model=finbert_classification_model,
    tokenizer=finbert_tokenizer,
    train_data=train_dataset,
    dev_data=val_dataset,
    layers_to_train=["classifier.", "bert.pooler.", "bert.encoder.layer.11.",
                     "bert.encoder.layer.10.", "bert.encoder.layer.9.", "bert.encoder.layer.8."],
    max_sequence_length=MAX_LEN,
    batch_size=8,
    num_epochs=10,
    # saved_output_dir="./finbert_truncation_test_3"
    saved_output_dir="/tmp/temp_model"
)

Map:   0%|          | 0/13786 [00:00<?, ? examples/s]

Map:   0%|          | 0/3447 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.523500,0.902430,0.507688
2,0.433400,1.045982,0.501015
3,0.365500,1.483322,0.510009
4,0.299400,1.857331,0.519872


In [ ]:
# Getting the predictions output
preprocessed_dev_data = val_dataset.map(preprocess_transcripts, batched=True, fn_kwargs={'tokenizer': finbert_tokenizer})
predictions_output = f_trainer.predict(preprocessed_dev_data)

# Getting the predicted class labels
preds = np.argmax(predictions_output.predictions, axis=1)
labels = predictions_output.label_ids

Map:   0%|          | 0/3447 [00:00<?, ? examples/s]

In [ ]:
# Getting the accuracy, precision, recall, and f1-scores
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

print("For the majority (up) class:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

For the majority (up) class:
Accuracy:  0.5199
Precision: 0.5382
Recall:    0.7422
F1 Score:  0.6239


The accuracy for this model still wasn't great, and while it did better on the recall than before it's still rather high. To make sure this isn't a technical issue, let's try recreating the FinBERT class and re-run things.


## Re-creating FinBERT class to manually change to binary output
One reason why things may not have worked previously is that our previous function simply dropped the 3-class classification head, so let's try doing it in the function manually.

*Note: I inadvertently froze nearly all the layers during this step (by forgetting to reset the model), so there was practically no training done here. I'm only keeping this here to have a record of experimenting with manually updating the model myself, though for the sake of the project, please ignore the following section and its results.

In [ ]:
# Loading the FinBERT model and tokenizer from Hugging face
finbert_model_name = "ProsusAI/finbert"
finbert_tokenizer = AutoTokenizer.from_pretrained(finbert_model_name)

In [ ]:
# Defining own subclass to replace the softmax output with a binary classification output
class FinBERTBinaryClassifier(BertPreTrainedModel):

    # Getting the previous class
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, 2)
        self.init_weights()

    # Defining the forward function to update the loss
    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {'loss': loss, 'logits': logits}

In [ ]:
# Loading the new configuration from our class
config = BertConfig.from_pretrained(finbert_model_name, num_labels=2)

# Loading the updated binary classifier model
finbert_binary_classifier = FinBERTBinaryClassifier.from_pretrained(
    finbert_model_name,
    config=config,
    ignore_mismatched_sizes=True)

Some weights of FinBERTBinaryClassifier were not initialized from the model checkpoint at ProsusAI/finbert and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Checking out the parameters
for name, param in finbert_binary_classifier.named_parameters():
    print(name, param.shape)

bert.embeddings.word_embeddings.weight torch.Size([30522, 768])
bert.embeddings.position_embeddings.weight torch.Size([512, 768])
bert.embeddings.token_type_embeddings.weight torch.Size([2, 768])
bert.embeddings.LayerNorm.weight torch.Size([768])
bert.embeddings.LayerNorm.bias torch.Size([768])
bert.encoder.layer.0.attention.self.query.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.self.query.bias torch.Size([768])
bert.encoder.layer.0.attention.self.key.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.self.key.bias torch.Size([768])
bert.encoder.layer.0.attention.self.value.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.self.value.bias torch.Size([768])
bert.encoder.layer.0.attention.output.dense.weight torch.Size([768, 768])
bert.encoder.layer.0.attention.output.dense.bias torch.Size([768])
bert.encoder.layer.0.attention.output.LayerNorm.weight torch.Size([768])
bert.encoder.layer.0.attention.output.LayerNorm.bias torch.Size([768])
bert.encoder

## FinBERT with truncated text 2
Once again, I'll get my truncated text and test things again.

*Note: Please note that the next two experiments were done while accidentally freezing nearly every layer, which is why it only predicted one class. Therefore, ignore the following two tests and skip to the third section of the truncated FinBERT testing. I'm only keeping this here to have a record of experimenting with manually updating the model myself - though it's clear something went awry.

In [ ]:
# Getting the max length for FinBERT
MAX_LEN = 512

# Splitting the data using train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    model_df['Text'].tolist(),
    model_df['Close_5_dir'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=model_df['Close_5_dir']
)

# Converting to dictionaries
train_dict = {'text': X_train, 'label': y_train}
val_dict = {'text': X_val, 'label': y_val}

# Creating HuggingFace Datasets
train_dataset = Dataset.from_dict(train_dict)
val_dataset = Dataset.from_dict(val_dict)

In [ ]:
# Defining a function for computing the metrics
metric = evaluate.load('accuracy')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# Defining a preprocessing transcripts function for my earnings data
def preprocess_transcripts(data, tokenizer):
    texts = data['text']

    encoded = tokenizer.batch_encode_plus(
        texts,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    return {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'label': data['label']
    }

In [ ]:
# Preprocessing the train and dev data outside the model
preprocessed_train_data = train_dataset.map(preprocess_transcripts, batched=True, fn_kwargs={'tokenizer': finbert_tokenizer})
preprocessed_dev_data = val_dataset.map(preprocess_transcripts, batched=True, fn_kwargs={'tokenizer': finbert_tokenizer})

Map:   0%|          | 0/13786 [00:00<?, ? examples/s]

Map:   0%|          | 0/3447 [00:00<?, ? examples/s]

In [ ]:
# Creating a general purpose classification model
def fine_tune_classif_model(classification_model,
                            preprocessed_train_data,
                            preprocessed_dev_data,
                            layers_to_train=["classifier."],
                            max_sequence_length=8192,
                            batch_size=8,
                            num_epochs=2,
                            saved_output_dir="/tmp/temp_model"):

    # Freezing all model parameters except those explicitly listed
    for name, param in classification_model.named_parameters():
        if not any(x in name for x in layers_to_train):
            param.requires_grad = False
        else:
            param.requires_grad = True

    training_args = TrainingArguments(
        output_dir=saved_output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        report_to="none",
        # load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True
    )

    trainer = Trainer(
        model=classification_model,
        args=training_args,
        train_dataset=preprocessed_train_data,
        eval_dataset=preprocessed_dev_data,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.001)]
    )

    trainer.train()
    return trainer

In [ ]:
# Setting torch to high for better performance
torch.set_float32_matmul_precision('high')

### FinBERT w/ truncated text test 2.1
Again, we'll start FinBERT with batch size of 8, num epochs of 10, and only unfreezing the classifier.

In [ ]:
# Calling the fine-tuning function and training on the model
f_trainer = fine_tune_classif_model(
    classification_model=finbert_binary_classifier,
    preprocessed_train_data=preprocessed_train_data,
    preprocessed_dev_data=preprocessed_dev_data,
    layers_to_train=["classifier."],
    max_sequence_length=MAX_LEN,
    batch_size=8,
    num_epochs=10,
    saved_output_dir="/tmp/temp_model"
)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.694200,0.690682,0.536408
2,0.695100,0.691775,0.534958
3,0.697500,0.691711,0.532927
4,0.692100,0.691283,0.536699


In [ ]:
# Getting the predictions output
predictions_output = f_trainer.predict(preprocessed_dev_data)

# Getting the predicted class labels
preds = np.argmax(predictions_output.predictions, axis=1)
labels = predictions_output.label_ids

In [ ]:
# Getting the accuracy, precision, recall, and f1-scores
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

print("For the majority (up) class:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

For the majority (up) class:
Accuracy:  0.5367
Precision: 0.5367
Recall:    1.0000
F1 Score:  0.6985


Creating my own subclass initiallly seems to have failed, as I'm getting 100% recall for the "Up" class. However, this is likely because I'm only unfreezing the classifier layer, and the model isn't being allowed to train as much.

Therefore, I'm next going to try unfreezing even more layers (including the embedding layers) to see if that makes a difference.

### FinBERT w/ truncated text test 2.2
This time, I'll try my model with unfreezing a large majority of the layers.

In [ ]:
# Calling the fine-tuning function and training on the model
f_trainer = fine_tune_classif_model(
    classification_model=finbert_binary_classifier,
    preprocessed_train_data=preprocessed_train_data,
    preprocessed_dev_data=preprocessed_dev_data,
    layers_to_train=["bert.embeddings.", "bert.encoder.layer.0.", "bert.encoder.layer.1.", "bert.encoder.layer.2.", "bert.encoder.layer.3.",
                     "bert.encoder.layer.8.", "bert.encoder.layer.9.", "bert.encoder.layer.10.", "bert.encoder.layer.11.",
                     "bert.pooler.", "classifier."],
    max_sequence_length=MAX_LEN,
    batch_size=8,
    num_epochs=10,
    saved_output_dir="/tmp/temp_model"
)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.698000,0.690531,0.536699
2,0.698600,0.691739,0.536699
3,0.695900,0.690903,0.536699
4,0.693300,0.690453,0.536699


In [ ]:
# Getting the predictions output
predictions_output = f_trainer.predict(preprocessed_dev_data)

# Getting the predicted class labels
preds = np.argmax(predictions_output.predictions, axis=1)
labels = predictions_output.label_ids

In [ ]:
# Getting the accuracy, precision, recall, and f1-scores
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

print("For the majority (up) class:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

For the majority (up) class:
Accuracy:  0.5367
Precision: 0.5367
Recall:    1.0000
F1 Score:  0.6985


At this point, there's no need to test this further, and it appears that the model itself is not learning well in the binary classification task (either that or something is wrong with my updated model). To try keeping the original 3-class softmax in tact, I'll try simply adding one more binary classification on top of the original softmax.

## Re-Loading in FinBERT and adding in binary layer again
One problem with the previous implementation of FinBERT is that we've simply gotten rid of the 3-class classification and replaced it with our own binary classification, which - especially for the second truncated test - likely caused the model to train incorrectly. We'll attempt to fix this by keeping the original 3-class softmax and adding our own binary classification on top of that. However, this really shouldn't change much since there's only one set of parameters with 3 nodes connecting the final two layers, and most of our training will be done in the transformer layers anyway.

In [ ]:
# Loading the FinBERT model and tokenizer from Hugging face
finbert_model_name = "ProsusAI/finbert"
finbert_tokenizer = AutoTokenizer.from_pretrained(finbert_model_name)

In [ ]:
# Defining own subclass to add a binary classification output on top of the the softmax output
class FinBERTWithBinaryHead(nn.Module):

    # Getting the previous class
    def __init__(self, pretrained_model_name=finbert_model_name):
        super().__init__()

        # Loading the pretrained FinBERT model for 3-class classification
        self.finbert = BertForSequenceClassification.from_pretrained(pretrained_model_name)

        # Adding a new binary classification head on top of the 3-class logits
        self.binary_head = nn.Linear(3, 2)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None):

        # Getting logits from original 3-class head
        outputs = self.finbert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        finbert_logits = outputs.logits

        # Feeding into the new binary head
        binary_logits = self.binary_head(finbert_logits)

        # Computing the loss
        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(binary_logits, labels)

        return {
            "loss": loss,
            "logits": binary_logits
        }

In [ ]:
# Loading the new configuration from our class
finbert_binary_model = FinBERTWithBinaryHead(pretrained_model_name=finbert_model_name)

In [ ]:
# Checking out the parameters
for name, param in finbert_binary_model.named_parameters():
    print(name, param.shape)

finbert.bert.embeddings.word_embeddings.weight torch.Size([30522, 768])
finbert.bert.embeddings.position_embeddings.weight torch.Size([512, 768])
finbert.bert.embeddings.token_type_embeddings.weight torch.Size([2, 768])
finbert.bert.embeddings.LayerNorm.weight torch.Size([768])
finbert.bert.embeddings.LayerNorm.bias torch.Size([768])
finbert.bert.encoder.layer.0.attention.self.query.weight torch.Size([768, 768])
finbert.bert.encoder.layer.0.attention.self.query.bias torch.Size([768])
finbert.bert.encoder.layer.0.attention.self.key.weight torch.Size([768, 768])
finbert.bert.encoder.layer.0.attention.self.key.bias torch.Size([768])
finbert.bert.encoder.layer.0.attention.self.value.weight torch.Size([768, 768])
finbert.bert.encoder.layer.0.attention.self.value.bias torch.Size([768])
finbert.bert.encoder.layer.0.attention.output.dense.weight torch.Size([768, 768])
finbert.bert.encoder.layer.0.attention.output.dense.bias torch.Size([768])
finbert.bert.encoder.layer.0.attention.output.LayerN

## FinBERT with truncated text 3
Once again, I'll get my truncated text and test things again.

In [ ]:
# Getting the max length for FinBERT
MAX_LEN = 512

# Splitting the data using train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    model_df['Text'].tolist(),
    model_df['Close_5_dir'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=model_df['Close_5_dir']
)

# Converting to dictionaries
train_dict = {'text': X_train, 'label': y_train}
val_dict = {'text': X_val, 'label': y_val}

# Creating HuggingFace Datasets
train_dataset = Dataset.from_dict(train_dict)
val_dataset = Dataset.from_dict(val_dict)

In [ ]:
# Defining a function for computing the metrics
metric = evaluate.load('accuracy')

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# Defining a preprocessing transcripts function for my earnings data
def preprocess_transcripts(data, tokenizer):
    texts = data['text']

    encoded = tokenizer.batch_encode_plus(
        texts,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    return {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'label': data['label']
    }

In [ ]:
# Preprocessing the train and dev data outside the model
preprocessed_train_data = train_dataset.map(preprocess_transcripts, batched=True, fn_kwargs={'tokenizer': finbert_tokenizer})
preprocessed_dev_data = val_dataset.map(preprocess_transcripts, batched=True, fn_kwargs={'tokenizer': finbert_tokenizer})

Map:   0%|          | 0/13786 [00:00<?, ? examples/s]

Map:   0%|          | 0/3447 [00:00<?, ? examples/s]

In [ ]:
# Creating a general purpose classification model
def fine_tune_classif_model(classification_model,
                            preprocessed_train_data,
                            preprocessed_dev_data,
                            layers_to_train=["classifier."],
                            max_sequence_length=8192,
                            batch_size=8,
                            num_epochs=2,
                            saved_output_dir="/tmp/temp_model"):

    # Freezing all model parameters except those explicitly listed
    for name, param in classification_model.named_parameters():
        if not any(x in name for x in layers_to_train):
            param.requires_grad = False
        else:
            param.requires_grad = True

    training_args = TrainingArguments(
        output_dir=saved_output_dir,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        report_to="none",
        # load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        greater_is_better=True
    )

    trainer = Trainer(
        model=classification_model,
        args=training_args,
        train_dataset=preprocessed_train_data,
        eval_dataset=preprocessed_dev_data,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5, early_stopping_threshold=0.001)]
    )

    trainer.train()
    return trainer

### FinBERT w/ truncated text test 3.1
Again, we'll start FinBERT with batch size of 8, num epochs of 10, and only unfreezing the pooler and final classifiers.

In [ ]:
# Calling the fine-tuning function and training on the model
f_trainer = fine_tune_classif_model(
    classification_model=finbert_binary_model,
    preprocessed_train_data=preprocessed_train_data,
    preprocessed_dev_data=preprocessed_dev_data,
    layers_to_train=["finbert.bert.pooler.", "finbert.classifier.", "binary_head."],
    max_sequence_length=MAX_LEN,
    batch_size=8,
    num_epochs=10,
    saved_output_dir="tmp/temp_model_3_1"
)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.693600,0.692191,0.535828
2,0.692300,0.695709,0.523934
3,0.692800,0.694037,0.529156
4,0.688900,0.692517,0.537279


In [ ]:
# Getting the predictions output
predictions_output = f_trainer.predict(preprocessed_dev_data)

# Getting the predicted class labels
preds = np.argmax(predictions_output.predictions, axis=1)
labels = predictions_output.label_ids

In [ ]:
# Getting the accuracy, precision, recall, and f1-scores
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

print("For the majority (up) class:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

For the majority (up) class:
Accuracy:  0.5274
Precision: 0.5342
Recall:    0.9319
F1 Score:  0.6791


The recall is still quite high, indicating there's still something that's happening with the learning.

On a different note, one reason this might be happening in the first place is because FinBERT was trained as a sentiment classifier, and earnings calls are generally going to be quite positive overall. Therefore, there may be an underlying reason why FinBERT is predicting so many positive results. One way to counteract this could be to train for a longer period, but another way might be to adjust the weights of the entire model. That way, the model may be able to learn the more nuanced, underlying tone that may be better for prediction. Later, I can also try analyzing the text (in chunks) using this FinBERT model and see if it's actually as positive as it seems.

### FinBERT w/ truncated text test 3.2
As mentioned above, I will try training the classifier again, but this time unfreezing every single layer to see if the model can learn better.

In [ ]:
# Calling the fine-tuning function and training on the model
f_trainer = fine_tune_classif_model(
    classification_model=finbert_binary_model,
    preprocessed_train_data=preprocessed_train_data,
    preprocessed_dev_data=preprocessed_dev_data,
    layers_to_train=["finbert.", "binary_head."],
    max_sequence_length=MAX_LEN,
    batch_size=8,
    num_epochs=10,
    saved_output_dir="tmp/temp_model_3_2"
)

Epoch,Training Loss,Validation Loss,Accuracy
1,0.691600,0.690904,0.536699
2,0.694600,0.691419,0.536699
3,0.694400,0.690466,0.536699
4,0.690900,0.690452,0.536699


In [ ]:
# Getting the predictions output
predictions_output = f_trainer.predict(preprocessed_dev_data)

# Getting the predicted class labels
preds = np.argmax(predictions_output.predictions, axis=1)
labels = predictions_output.label_ids

In [ ]:
# Getting the accuracy, precision, recall, and f1-scores
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

print("For the majority (up) class:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

For the majority (up) class:
Accuracy:  0.5367
Precision: 0.5367
Recall:    1.0000
F1 Score:  0.6985


Unfortunately, it appears the previous training kept most of its parameters, and it did not improve the model at all.

### FinBERT w/ truncated text test 3.3
I'll try one more thing, which is to only change the parameters like I did in my very first test (along with my additional binary_head) and see if I can get the recall to come down.

Also, I'll try reinitializing the model so it can start training from scratch.

***Note**: I changed early stopping to 5 steps with a lower accuracy threshold; I also removed loading the best model so we can get a better idea of why the recall is so high.

In [ ]:
# Reinitializing the model
finbert_binary_model = FinBERTWithBinaryHead(pretrained_model_name=finbert_model_name)

In [ ]:
# Calling the fine-tuning function and training on the model
f_trainer = fine_tune_classif_model(
    classification_model=finbert_binary_model,
    preprocessed_train_data=preprocessed_train_data,
    preprocessed_dev_data=preprocessed_dev_data,
    layers_to_train=["binary_head.", "classifier.", "bert.pooler.", "bert.encoder.layer.11."],
    max_sequence_length=MAX_LEN,
    batch_size=8,
    num_epochs=10,
    saved_output_dir="tmp/temp_model_3_3"
)

Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.697600,0.690529,0.536699
2,0.697700,0.690926,0.536699
3,0.698200,0.690196,0.536989
4,0.690400,0.691541,0.534958
5,0.690100,0.713070,0.532637
6,0.684600,0.707380,0.521613


In [ ]:
# Getting the predictions output
predictions_output = f_trainer.predict(preprocessed_dev_data)

# Getting the predicted class labels
preds = np.argmax(predictions_output.predictions, axis=1)
labels = predictions_output.label_ids

In [ ]:
# Getting the accuracy, precision, recall, and f1-scores
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')

print("For the majority (up) class:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

For the majority (up) class:
Accuracy:  0.5216
Precision: 0.5411
Recall:    0.7146
F1 Score:  0.6159


After performing more tests such as increasing the epochs required for early stopping (5), decreasing the accuracy threshold (0.01), and not reloading the best model, it has become clear what's happening. Essentially, as the training loss improves, the model continues to overfit, which affects validation loss and therefore the validation accuracy. This implies that the model is already doing the best it can in the beginning, and for one reason or another, it can't seem to improve from there.

This can be seen when we compare the last epoch vs the best epoch (when load_best_model_at_end is true). Specifically, loading the best model means loading the model with the highest accuracy, which happens to be the majority class predictor essentially. Therefore, that's why we see such high recall for the majority class. When loading the most *recent* model based on the epoch, we achieve the more appropriate understanding that the model is simply overfitting.

In either case, this method clearly does not work well, and if we had to choose one of these FinBERT models, we should stick with the original implementation of FinBERT (i.e., directly replacing the softmax with the binary classification). It's slightly less complex, and it appears to perform slightly better.

However, let's try performing some other tests, specifically with chunked earnings transcripts, along with checking how positive the sentiments actually are in the earnings call (by section). For the latter, if most of the sentiment is positive anyway, then using a sentiment classifier such as FinBERT was unfortunately doomed from the start.